# P3 · Notebook 03a: Does the retry system still work on the low-memory model? (GPU, ~30–45 min)

**Why this notebook exists:** notebook 02b measured the router + quality-gated cascade with the **plain float16** model (~8 GB of GPU memory). The app will run the **low-memory** model from notebook 01 (~3 GB). Quantizing can change pictures slightly, so this notebook re-measures everything on the low-memory model, using 02b's **test prompts**, **seeds**, **gate**, **router** and **independent judge**. Then every number we quote for the app is measured on the setup the app actually runs.

**What it does:**
1. Loads 02b's results from Drive: test prompts, picture-by-picture results, router, gate settings.
2. Loads the low-memory model + the gate **exactly as the app does** (same code: `p3_engine.py`), and measures speed per level and the **whole app's GPU memory**.
3. Draws every test prompt at every level with the low-memory model; the gate and the judge score every picture.
4. Replays the strategies for **both** models side by side, and checks that our exported router reproduces 02b's numbers.
5. Measures the app's three modes (Fast / Balanced / Quality), including a few live requests.
6. Saves `cascade_config_lowmem.json` for the app.

**Before running:** 02b must have finished (its last cell printed `Saved: cascade_config.json`). Runtime → **T4 GPU**. Progress is saved to Drive: if Colab disconnects, **Run all** again.

## 1 · Settings

In [ ]:
import os
LOWMEM_SETUP = "M3"    # "M3" = 4-bit reader + 4-bit drawer (smallest, 2.86 GB in notebook 01) | "M2" = 8-bit drawer
MODE_TAG = None        # None = use what 02b produced ("full" if it exists, otherwise "quick")
SEED = 0               # must match 02b
JUDGE_CLIP = "openai/clip-vit-large-patch14"
N_LIVE = 10            # live requests per app mode at the end (full pipeline incl. polish + WebP)
SAVE_TO_DRIVE = True   # Colab: read 02b from Drive and save there. On Kaggle this switches itself off.
GITHUB_REPO = "Puneetha-S-Shankar/text-image-CV"   # Kaggle: 02b's results come from this repo (results/02b)

## 2 · Install (~1–2 min)

In [ ]:
import subprocess, sys
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-U", "diffusers", "transformers", "accelerate",
                "bitsandbytes", "scikit-learn", "tabulate", "sentencepiece", "protobuf"], check=True)
print("Libraries installed.")

## 3 · Our modules
The same files as `src/` in the repo. `p3_engine.py` is the code the app runs: generator loading, the gate, the router and the cascade.

In [ ]:
%%writefile p3_classical.py
"""p3_classical.py: the classical computer-vision layer of our P3 text-to-image system.

No neural networks in here, only image maths (OpenCV + NumPy). This layer judges VISUAL quality only.
Whether the picture matches the prompt (semantic quality) is checked elsewhere (CLIP + detector, notebook 02b).

    assess(img, prompt)        no-reference visual check -> metrics, flags, verdict ("ok" / "fix" / "regenerate")
    safe_enhance(img, prompt)  fix only what was flagged, re-check, and KEEP THE FIX ONLY IF IT HELPED
    quality_maps(img)          per-region detail / noise maps (so fixes touch only the regions that need them)
    upscale(img, size)         classical resizing, incl. an edge-aware upscaler (Lanczos + back-projection)
    pack_for_mobile(img)       resize + smallest WebP/JPEG that keeps SSIM >= target
    psnr(a, b), ssim(a, b)     reference metrics for the experiments

Images are RGB uint8 numpy arrays (H, W, 3). to_rgb() converts PIL images.
"""
from __future__ import annotations

import time

import cv2
import numpy as np

ANALYSIS_SIZE = 512  # checks run on a copy with this longest side, so thresholds don't depend on resolution

# Thresholds, calibrated on real photos + synthetic damage (notebook 02a re-validates them on held-out images).
TH = {
    "blank_std": 4.0,           # brightness spread below this -> blank / broken
    "dark_mean": 90.0,          # too dark: mean brightness below this ...
    "dark_p95": 135.0,          # ... AND even the brightest 5% are dim
    "bright_mean": 170.0,       # too bright / washed out: mean above this ...
    "bright_p5": 110.0,         # ... AND even the darkest 5% are light
    "low_contrast": 0.10,       # RMS contrast (std / 255) below this -> flat
    "blur": 0.04,               # contrast-normalised sharpness below this -> blurry
    "noise_sigma": 3.8,         # noise estimated in flat areas (grey levels) above this -> grainy
    "cast_gray": 13.0,          # colour shift of the near-grey pixels above this -> tint. Deliberately conservative:
                                # warm scenes (brown fur, lamp light) also score high, so only strong tints count
    "periodic": 2.5,            # FFT spike strength above this -> grid / checkerboard glitch
}

# Do-no-harm guard: a fix may not raise the noise estimate by more than this (grey levels), e.g. CLAHE/sharpening on sky.
MAX_ADDED_GRAIN = 1.5   # brightening alone adds ~+0.3 (reveals shadow noise); gamma+CLAHE can reach +2

# Problems a classical fix can't really undo: if they remain, the cascade should generate again with more steps.
UNFIXABLE = {"blank", "blurry", "noisy", "periodic_artifact"}

# Words in the prompt that mean "this look is intended, don't fix it".
INTENT = {
    "too_dark": ["night", "dark", "noir", "silhouette", "midnight", "shadow", "candlelit", "moody", "low light", "starry"],
    "too_bright": ["bright white", "overexposed", "high key", "snow", "foggy white", "minimalist white"],
    "low_contrast": ["fog", "foggy", "mist", "haze", "hazy", "pastel", "soft light", "dreamy", "minimalist"],
    "colour_cast": ["sunset", "sunrise", "golden", "neon", "sepia", "monochrome", "black and white",
                    "underwater", "vintage", "duotone", "red light", "blue light", "candle", "warm", "autumn",
                    "fire", "desert", "lamp", "cozy", "cosy", "tungsten", "night city"],
    "blurry": ["bokeh", "blurry", "blurred", "motion blur", "out of focus", "depth of field", "dreamy",
               "fog", "foggy", "mist", "long exposure", "soft focus"],
    "noisy": ["film grain", "grainy", "noise", "noisy", "vintage photo", "analog"],
}


# ----------------------------------------------------------------------------- helpers
def to_rgb(img) -> np.ndarray:
    """PIL image / array -> RGB uint8 array (H, W, 3)."""
    if hasattr(img, "convert"):
        return np.asarray(img.convert("RGB")).copy()
    a = np.asarray(img)
    if a.ndim == 2:
        a = np.stack([a] * 3, axis=-1)
    if a.dtype != np.uint8:
        a = np.clip(a, 0, 255).astype(np.uint8)
    return np.ascontiguousarray(a[..., :3])


def _analysis_view(rgb: np.ndarray) -> np.ndarray:
    h, w = rgb.shape[:2]
    s = ANALYSIS_SIZE / max(h, w)
    if s < 1:
        rgb = cv2.resize(rgb, (round(w * s), round(h * s)), interpolation=cv2.INTER_AREA)
    return rgb


def _gray(rgb: np.ndarray) -> np.ndarray:
    return cv2.cvtColor(rgb, cv2.COLOR_RGB2GRAY)


def _edge_strength(gray: np.ndarray) -> np.ndarray:
    g = gray.astype(np.float32)
    mag = cv2.magnitude(cv2.Sobel(g, cv2.CV_32F, 1, 0, ksize=3), cv2.Sobel(g, cv2.CV_32F, 0, 1, ksize=3))
    return cv2.GaussianBlur(mag, (0, 0), 1.5)


# ----------------------------------------------------------------------------- metrics
def sharpness(gray: np.ndarray) -> float:
    """Variance of the Laplacian divided by brightness variance: high = crisp edges.
    Dividing by contrast means a dark or flat picture isn't wrongly called blurry."""
    g = gray.astype(np.float64)
    return float(cv2.Laplacian(g, cv2.CV_64F).var() / (g.var() + 1e-6))


def noise_sigma(gray: np.ndarray) -> float:
    """Immerkaer (1996) noise estimate, measured only in the flattest half of the picture
    (so texture and edges are not mistaken for noise). Unit: grey levels."""
    g = gray.astype(np.float64)
    k = np.array([[1, -2, 1], [-2, 4, -2], [1, -2, 1]], np.float64)
    resp = np.abs(cv2.filter2D(g, -1, k, borderType=cv2.BORDER_REFLECT))[1:-1, 1:-1]
    edges = _edge_strength(gray)[1:-1, 1:-1]
    flat = edges <= np.percentile(edges, 50)
    return float(np.sqrt(np.pi / 2) * resp[flat].mean() / 6.0)


def colourfulness(rgb: np.ndarray) -> float:
    """Hasler & Suesstrunk (2003) colourfulness: 0 = grey, ~30 = normal photo, 60+ = very colourful."""
    r, g, b = (rgb[..., i].astype(np.float64) for i in range(3))
    rg, yb = r - g, 0.5 * (r + g) - b
    return float(np.hypot(rg.std(), yb.std()) + 0.3 * np.hypot(rg.mean(), yb.mean()))


def colour_cast(rgb: np.ndarray) -> tuple[float, float]:
    """Tint test in LAB on the NEAR-GREY pixels (the least colourful 15%, ignoring very dark/bright ones).
    In a normal picture those pixels are close to neutral, even if the scene is colourful (a red car, green grass).
    If even they are pushed towards one colour, the whole picture has a tint.
    Returns (shift of near-grey pixels, average shift of all pixels), both in LAB units."""
    lab = cv2.cvtColor(rgb, cv2.COLOR_RGB2LAB).astype(np.float64)
    L, a, b = lab[..., 0], lab[..., 1] - 128.0, lab[..., 2] - 128.0
    chroma = np.hypot(a, b)
    mid = (L > 40) & (L < 235)
    if mid.sum() < 100:
        mid = np.ones_like(L, bool)
    grayish = mid & (chroma <= np.percentile(chroma[mid], 15))
    gray_shift = float(np.hypot(a[grayish].mean(), b[grayish].mean()))
    return gray_shift, float(np.hypot(a.mean(), b.mean()))


def _spectrum_residual(gray: np.ndarray):
    """Log-magnitude spectrum minus its smooth background. Isolated spikes = periodic patterns."""
    g = gray.astype(np.float32)
    g = g - g.mean()
    h, w = g.shape
    win = np.outer(np.hanning(h), np.hanning(w)).astype(np.float32)
    spec = np.log1p(np.abs(np.fft.fftshift(np.fft.fft2(g * win)))).astype(np.float32)
    bg = cv2.GaussianBlur(spec, (0, 0), 3.0, borderType=cv2.BORDER_REFLECT_101)
    yy, xx = np.ogrid[:h, :w]
    r = np.hypot((yy - h // 2) / (h / 2), (xx - w // 2) / (w / 2))  # 0 = DC, 1 = Nyquist
    return spec - bg, r > 0.25  # ignore low frequencies (normal picture content lives there)


def periodic_score(gray: np.ndarray) -> float:
    resid, mask = _spectrum_residual(gray)
    return float(resid[mask].max())


# ----------------------------------------------------------------------------- 1. assess
def _intended(flag: str, prompt: str | None) -> bool:
    p = (prompt or "").lower()
    return any(word in p for word in INTENT.get(flag, []))


def assess(img, prompt: str | None = None) -> dict:
    """Visual quality check. Returns metrics, flags, ignored (flags the prompt asked for), verdict, qa_score."""
    rgb = _analysis_view(to_rgb(img))
    gray = _gray(rgb)
    g = gray.astype(np.float64)
    cast_gray, cast_all = colour_cast(rgb)
    m = {
        "mean": g.mean(), "p5": np.percentile(g, 5), "p95": np.percentile(g, 95), "std": g.std(),
        "contrast_rms": g.std() / 255.0, "sharpness": sharpness(gray), "noise_sigma": noise_sigma(gray),
        "colourfulness": colourfulness(rgb), "cast_gray": cast_gray, "cast_all": cast_all,
        "periodic": periodic_score(gray),
    }
    raw = []
    if m["std"] < TH["blank_std"]:
        raw.append("blank")
    else:
        if m["mean"] < TH["dark_mean"] and m["p95"] < TH["dark_p95"]:
            raw.append("too_dark")
        if m["mean"] > TH["bright_mean"] and m["p5"] > TH["bright_p5"]:
            raw.append("too_bright")
        if m["contrast_rms"] < TH["low_contrast"]:
            raw.append("low_contrast")
        if m["sharpness"] < TH["blur"]:
            raw.append("blurry")
        if m["noise_sigma"] > TH["noise_sigma"]:
            raw.append("noisy")
        if m["cast_gray"] > TH["cast_gray"]:
            raw.append("colour_cast")
        if m["periodic"] > TH["periodic"]:
            raw.append("periodic_artifact")
    flags = [f for f in raw if f == "blank" or not _intended(f, prompt)]
    ignored = [f for f in raw if f not in flags]
    verdict = "regenerate" if "blank" in flags else ("fix" if flags else "ok")
    return {
        "metrics": {key: float(v) for key, v in m.items()},
        "flags": flags,
        "ignored_by_prompt": ignored,
        "verdict": verdict,
        "qa_score": 0 if "blank" in flags else max(0, 100 - 15 * len(flags)),
    }


# ----------------------------------------------------------------------------- 2. corrections
def guided_filter(guide: np.ndarray, src: np.ndarray, r: int, eps: float) -> np.ndarray:
    """He et al. (2010) guided filter: edge-preserving smoothing in O(N). guide: gray uint8, src: uint8 (H,W[,3]).
    Returns float32 in [0, 1]."""
    I = guide.astype(np.float32) / 255.0
    p = src.astype(np.float32) / 255.0
    if p.ndim == 3:
        I = I[..., None]

    def box(x):
        y = cv2.boxFilter(x, -1, (2 * r + 1, 2 * r + 1), borderType=cv2.BORDER_REFLECT)
        return y[..., None] if (x.ndim == 3 and y.ndim == 2) else y

    mean_I, mean_p = box(I), box(p)
    a = (box(I * p) - mean_I * mean_p) / (box(I * I) - mean_I * mean_I + eps)
    b = mean_p - a * mean_I
    return box(a) * I + box(b)


def remove_periodic(rgb: np.ndarray, thresh: float | None = None) -> tuple[np.ndarray, int]:
    """Notch filter: find spikes in the frequency spectrum and push them down to the background level."""
    thresh = TH["periodic"] if thresh is None else thresh
    resid, mask = _spectrum_residual(_gray(rgb))
    peaks = (resid > thresh) & mask
    n = int(peaks.sum())
    if n == 0:
        return rgb, 0
    peaks = np.fft.ifftshift(cv2.dilate(peaks.astype(np.uint8), np.ones((5, 5), np.uint8)) > 0)
    out = np.empty(rgb.shape, np.float32)
    for c in range(3):
        F = np.fft.fft2(rgb[..., c].astype(np.float32))
        logmag = np.log1p(np.abs(F)).astype(np.float32)
        excess = logmag - cv2.GaussianBlur(logmag, (0, 0), 3.0, borderType=cv2.BORDER_REFLECT_101)
        # each channel's own spike height (measured without the window) is pushed down to its local background
        atten = np.where(peaks, np.exp(-np.clip(excess, 0, None)), 1.0)
        out[..., c] = np.real(np.fft.ifft2(F * atten))
    return np.clip(out, 0, 255).astype(np.uint8), n


def denoise(rgb: np.ndarray) -> tuple[np.ndarray, float]:
    """Edge-preserving bilateral filter, applied mainly in flat regions (textures and edges are kept)."""
    sigma = noise_sigma(_gray(rgb))
    den = cv2.bilateralFilter(rgb, 7, sigmaColor=max(10.0, 3.0 * sigma), sigmaSpace=3)
    den = cv2.bilateralFilter(den, 5, sigmaColor=max(10.0, 2.0 * sigma), sigmaSpace=2)
    e = _edge_strength(_gray(den))   # edges measured on the cleaned picture, so noise isn't mistaken for edges
    flat = 1.0 - np.clip(e / (np.percentile(e, 90) + 1e-6), 0, 1)
    out = flat[..., None] * den.astype(np.float32) + (1 - flat[..., None]) * rgb.astype(np.float32)
    return np.clip(out, 0, 255).astype(np.uint8), sigma


def white_balance(rgb: np.ndarray, strength: float = 0.8) -> np.ndarray:
    """Gray-pixel white balance (a gray-world variant) in LAB: measure the tint on the near-grey pixels only,
    then shift the whole picture back by `strength` of it. Colourful objects don't fool it."""
    lab = cv2.cvtColor(rgb, cv2.COLOR_RGB2LAB).astype(np.float32)
    L, a, b = lab[..., 0], lab[..., 1] - 128.0, lab[..., 2] - 128.0
    chroma = np.hypot(a, b)
    mid = (L > 40) & (L < 235)
    if mid.sum() < 100:
        mid = np.ones_like(L, bool)
    grayish = mid & (chroma <= np.percentile(chroma[mid], 15))
    lab[..., 1] -= strength * a[grayish].mean()
    lab[..., 2] -= strength * b[grayish].mean()
    return cv2.cvtColor(np.clip(lab, 0, 255).astype(np.uint8), cv2.COLOR_LAB2RGB)


def auto_gamma(rgb: np.ndarray, target: float = 0.45) -> tuple[np.ndarray, float]:
    """Gamma on brightness only (LAB L channel) so the average lands near `target`. Colours are kept."""
    lab = cv2.cvtColor(rgb, cv2.COLOR_RGB2LAB)
    mean = max(lab[..., 0].mean() / 255.0, 1e-3)
    gamma = float(np.clip(np.log(target) / np.log(mean), 0.45, 2.2))
    lut = (255.0 * (np.arange(256) / 255.0) ** gamma).clip(0, 255).astype(np.uint8)
    lab[..., 0] = cv2.LUT(lab[..., 0], lut)
    return cv2.cvtColor(lab, cv2.COLOR_LAB2RGB), gamma


def clahe(rgb: np.ndarray, clip: float = 2.0, tiles: int = 8) -> np.ndarray:
    """CLAHE (local contrast) on brightness only."""
    lab = cv2.cvtColor(rgb, cv2.COLOR_RGB2LAB)
    lab[..., 0] = cv2.createCLAHE(clipLimit=clip, tileGridSize=(tiles, tiles)).apply(lab[..., 0])
    return cv2.cvtColor(lab, cv2.COLOR_LAB2RGB)


def edge_aware_sharpen(rgb: np.ndarray, amount: float) -> np.ndarray:
    """Boost detail ONLY around edges: detail = image - guided-filter base, weighted by an edge mask.
    Flat areas (sky, skin) are left alone, so noise isn't amplified and halos stay small."""
    gray = _gray(rgb)
    base = guided_filter(gray, rgb, r=2, eps=1e-3)
    img = rgb.astype(np.float32) / 255.0
    e = _edge_strength(gray)
    mask = np.clip(e / (np.percentile(e, 95) + 1e-6), 0, 1)[..., None]
    return np.clip((img + amount * (img - base) * mask) * 255.0, 0, 255).astype(np.uint8)


def _apply_fixes(rgb: np.ndarray, rep: dict, stages: list | None, prompt: str | None):
    m, flags, actions, out = rep["metrics"], rep["flags"], [], rgb

    def log(name, img):
        actions.append(name)
        if stages is not None:
            stages.append((name, img))

    if "periodic_artifact" in flags:
        out, n = remove_periodic(out)
        log(f"notch filter (removed {n} spectrum spikes)", out)
        rep = assess(out, prompt)          # a grid also looks like noise: re-check before the next fixes
        m, flags = rep["metrics"], rep["flags"]
    if "noisy" in flags:
        out, sigma = denoise(out)
        log(f"edge-preserving denoise (noise sigma {sigma:.1f})", out)
    if "colour_cast" in flags:
        out = white_balance(out, 0.6)
        log(f"gray-pixel white balance (tint {m['cast_gray']:.1f} LAB units)", out)
    if "too_dark" in flags or "too_bright" in flags:
        out, gamma = auto_gamma(out)
        log(f"gamma on brightness (gamma {gamma:.2f})", out)
    # CLAHE for flat pictures. Dark pictures get it only if they're STILL flat after gamma:
    # on dark pictures CLAHE mostly amplifies grain in smooth areas (sky, walls).
    contrast_now = _gray(out).std() / 255.0
    if "low_contrast" in flags or contrast_now < TH["low_contrast"]:
        clip = float(np.clip(1.5 + 15 * (TH["low_contrast"] - contrast_now), 1.5, 3.0))
        out = clahe(out, clip)
        log(f"CLAHE local contrast (clip {clip:.1f})", out)
    if "blurry" in flags:
        amount = float(np.clip(0.6 + 2.0 * (TH["blur"] - m["sharpness"]) / TH["blur"], 0.6, 1.8))
        out = edge_aware_sharpen(out, amount)
        log(f"edge-aware sharpen (amount {amount:.2f})", out)
    return out, actions


def safe_enhance(img, prompt: str | None = None, return_stages: bool = False) -> tuple[np.ndarray, dict]:
    """Fix only what assess() flagged, then RE-CHECK. Keep the fix only if
    (1) at least one flagged problem is gone or clearly better, and (2) no new problem appeared.
    Otherwise return the original picture untouched ("do no harm")."""
    t0 = time.perf_counter()
    rgb = to_rgb(img)
    before = assess(rgb, prompt)
    stages = [("input", rgb)] if return_stages else None
    info = {"before": before, "after": None, "actions": [], "accepted": False, "decision": "",
            "escalate": False, "stages": stages}

    if before["verdict"] == "regenerate":
        info.update(escalate=True, decision="blank/broken: nothing to fix, ask the generator for a new picture")
    elif before["verdict"] == "ok":
        info["decision"] = "no visual problems: original kept"
    else:
        out, actions = _apply_fixes(rgb, before, stages, prompt)
        after = assess(out, prompt)
        fixed = [f for f in before["flags"] if f not in after["flags"]]
        new = [f for f in after["flags"] if f not in before["flags"]]
        grain_added = after["metrics"]["noise_sigma"] - before["metrics"]["noise_sigma"]
        if "noisy" not in before["flags"] and grain_added > MAX_ADDED_GRAIN:
            new.append(f"added_grain(+{grain_added:.1f})")   # a fix must not make smooth areas grainy
        info.update(after=after, actions=actions)
        if fixed and not new:
            info.update(accepted=True, decision=f"fix kept: solved {fixed}")
            rgb = out
            remaining = after["flags"]
        else:
            info["decision"] = f"fix rejected (solved {fixed}, new problems {new}): original kept"
            remaining = before["flags"]
        # Classical CV can't undo real blur, heavy noise or broken structure -> the cascade should spend more steps.
        unfixable = [f for f in remaining if f in UNFIXABLE]
        if unfixable:
            info.update(escalate=True, decision=info["decision"] + f"; still {unfixable}: escalate to a higher rung")
    info["ms"] = (time.perf_counter() - t0) * 1000
    return rgb, info


# ----------------------------------------------------------------------------- 3. region maps (visualisation)
def quality_maps(img, tile: int = 32) -> dict:
    """Per-tile detail (sharpness) and noise maps on the analysis-size picture, for 'where is it bad?' views."""
    gray = _gray(_analysis_view(to_rgb(img)))
    h, w = gray.shape
    rows, cols = h // tile, w // tile
    detail, noise = np.zeros((rows, cols)), np.zeros((rows, cols))
    lap = cv2.Laplacian(gray.astype(np.float64), cv2.CV_64F)
    for i in range(rows):
        for j in range(cols):
            sl = (slice(i * tile, (i + 1) * tile), slice(j * tile, (j + 1) * tile))
            detail[i, j] = lap[sl].var()
            noise[i, j] = noise_sigma(gray[sl])
    return {"detail": detail, "noise": noise}


# ----------------------------------------------------------------------------- 4. resizing
_INTERP = {"nearest": cv2.INTER_NEAREST, "bilinear": cv2.INTER_LINEAR,
           "bicubic": cv2.INTER_CUBIC, "lanczos": cv2.INTER_LANCZOS4}


def upscale(img, size: tuple[int, int], method: str = "edge_aware", iters: int = 4) -> np.ndarray:
    """Resize to size=(width, height). 'edge_aware' = Lanczos + iterative back-projection
    (Irani & Peleg 1991): repeatedly fix the big picture so that shrinking it gives back the small one."""
    rgb = to_rgb(img)
    W, H = size
    if method in _INTERP:
        return cv2.resize(rgb, (W, H), interpolation=_INTERP[method])
    if method != "edge_aware":
        raise ValueError(f"unknown method {method!r}")
    h, w = rgb.shape[:2]
    lr = rgb.astype(np.float32)
    x = cv2.resize(lr, (W, H), interpolation=cv2.INTER_LANCZOS4)
    for _ in range(iters):
        err = lr - cv2.resize(x, (w, h), interpolation=cv2.INTER_AREA)
        x += cv2.resize(err, (W, H), interpolation=cv2.INTER_CUBIC)
    return np.clip(x, 0, 255).astype(np.uint8)


# ----------------------------------------------------------------------------- 5. reference metrics
def psnr(a, b) -> float:
    mse = np.mean((to_rgb(a).astype(np.float64) - to_rgb(b).astype(np.float64)) ** 2)
    return float("inf") if mse == 0 else float(10 * np.log10(255.0 ** 2 / mse))


def ssim(a, b) -> float:
    """Standard SSIM (Wang et al. 2004) on brightness, 11x11 Gaussian window."""
    x, y = (_gray(to_rgb(i)).astype(np.float64) for i in (a, b))
    c1, c2 = (0.01 * 255) ** 2, (0.03 * 255) ** 2

    def blur(z):
        return cv2.GaussianBlur(z, (11, 11), 1.5)

    mx, my = blur(x), blur(y)
    vx, vy, cxy = blur(x * x) - mx * mx, blur(y * y) - my * my, blur(x * y) - mx * my
    return float((((2 * mx * my + c1) * (2 * cxy + c2)) / ((mx * mx + my * my + c1) * (vx + vy + c2))).mean())


# ----------------------------------------------------------------------------- 6. packing for phones
def encode(img, fmt: str = "webp", quality: int = 85) -> bytes:
    bgr = cv2.cvtColor(to_rgb(img), cv2.COLOR_RGB2BGR)
    fmt = fmt.lower()
    if fmt == "png":
        ok, buf = cv2.imencode(".png", bgr, [cv2.IMWRITE_PNG_COMPRESSION, 6])
    elif fmt in ("jpg", "jpeg"):
        ok, buf = cv2.imencode(".jpg", bgr, [cv2.IMWRITE_JPEG_QUALITY, int(quality)])
    elif fmt == "webp":
        ok, buf = cv2.imencode(".webp", bgr, [cv2.IMWRITE_WEBP_QUALITY, int(quality)])
    else:
        raise ValueError(f"unknown format {fmt!r}")
    if not ok:
        raise RuntimeError(f"encoding to {fmt} failed")
    return buf.tobytes()


def decode(data: bytes) -> np.ndarray:
    return cv2.cvtColor(cv2.imdecode(np.frombuffer(data, np.uint8), cv2.IMREAD_COLOR), cv2.COLOR_BGR2RGB)


def pack_for_mobile(img, max_side: int = 1024, fmt: str = "webp", min_ssim: float = 0.95,
                    q_lo: int = 30, q_hi: int = 95) -> tuple[bytes, dict]:
    """Shrink to the phone's size, then binary-search the SMALLEST quality whose SSIM >= min_ssim.
    SSIM is measured on a <=512px copy to keep the search fast (~5 encodes)."""
    t0 = time.perf_counter()
    rgb = to_rgb(img)
    h, w = rgb.shape[:2]
    if max(h, w) > max_side:
        s = max_side / max(h, w)
        rgb = cv2.resize(rgb, (round(w * s), round(h * s)), interpolation=cv2.INTER_AREA)
    # Search the quality on a <=512px copy (fast), then encode the full-size picture once.
    ref = _analysis_view(rgb)
    best_q = q_hi
    lo, hi = q_lo, q_hi
    while lo <= hi:
        q = (lo + hi) // 2
        if ssim(ref, decode(encode(ref, fmt, q))) >= min_ssim:
            best_q, hi = q, q - 2
        else:
            lo = q + 2
    q = best_q
    data = encode(rgb, fmt, q)
    s_val = ssim(ref, _analysis_view(decode(data)))
    return data, {"format": fmt, "quality": q, "kb": len(data) / 1024, "ssim": s_val,
                  "size": rgb.shape[1::-1], "ms": (time.perf_counter() - t0) * 1000}

In [ ]:
%%writefile p3_semantic.py
"""p3_semantic.py: the SEMANTIC half of our quality gate. Does the picture show what the prompt asked for?

Follows GenEval's protocol (Ghosh et al., NeurIPS 2023) with lighter, pip-installable models:
  - objects + counts : COCO instance detector (torchvision Mask R-CNN v2) instead of mmdet Mask2Former
  - colours          : CLIP ViT-L/14 zero-shot on each object's masked crop (GenEval's prompts and 10 colours)
  - positions        : GenEval's relative_position() rule on box centres
  - overall match    : CLIP image-text score (same formula as notebook 01)
Our numbers are therefore "GenEval-style", NOT official GenEval scores.

    parse_prompt(text)                 -> checklist in GenEval's metadata format, or None if nothing checkable
    prompt_features(text)              -> cheap numeric clues about prompt difficulty (for the router)
    SemanticChecker(device)
        .clip_scores(images, prompts)  -> CLIP match per image (x100)
        .detect(image)                 -> {class: [{"box", "mask", "score"}, ...]} sorted by confidence
        .check(image, spec)            -> {"passed", "reasons", ...} GenEval-style structured check
        .gate(image, prompt, clip_min) -> the semantic verdict used by the cascade

Only the 80 COCO object types (GenEval's list) can be checked structurally. Other prompts fall back to CLIP.
"""
from __future__ import annotations

import re
import time
from collections import Counter

import numpy as np

# ----------------------------------------------------------------------------- vocabulary (GenEval)
CLASSES = [
    "person", "bicycle", "car", "motorcycle", "airplane", "bus", "train", "truck", "boat", "traffic light",
    "fire hydrant", "stop sign", "parking meter", "bench", "bird", "cat", "dog", "horse", "sheep", "cow",
    "elephant", "bear", "zebra", "giraffe", "backpack", "umbrella", "handbag", "tie", "suitcase", "frisbee",
    "skis", "snowboard", "sports ball", "kite", "baseball bat", "baseball glove", "skateboard", "surfboard",
    "tennis racket", "bottle", "wine glass", "cup", "fork", "knife", "spoon", "bowl", "banana", "apple",
    "sandwich", "orange", "broccoli", "carrot", "hot dog", "pizza", "donut", "cake", "chair", "couch",
    "potted plant", "bed", "dining table", "toilet", "tv", "laptop", "computer mouse", "tv remote",
    "computer keyboard", "cell phone", "microwave", "oven", "toaster", "sink", "refrigerator", "book", "clock",
    "vase", "scissors", "teddy bear", "hair drier", "toothbrush",
]
COLOURS = ["red", "orange", "yellow", "green", "blue", "purple", "pink", "brown", "black", "white"]
NUMBER_WORDS = {"two": 2, "three": 3, "four": 4, "five": 5, "six": 6, "seven": 7, "eight": 8, "nine": 9, "ten": 10}
ONE_WORDS = {"a", "an", "one", "single", "lone"}

# Everyday words -> COCO class. (Deliberately no "mouse": the animal would be mistaken for a computer mouse.)
SYNONYMS = {
    "man": "person", "woman": "person", "boy": "person", "girl": "person", "child": "person", "kid": "person",
    "lady": "person", "guy": "person", "astronaut": "person", "chef": "person",
    "bike": "bicycle", "motorbike": "motorcycle", "scooter": "motorcycle", "plane": "airplane",
    "aeroplane": "airplane", "jet": "airplane", "ship": "boat", "lorry": "truck", "puppy": "dog", "kitten": "cat",
    "sofa": "couch", "television": "tv", "smartphone": "cell phone", "phone": "cell phone",
    "mobile phone": "cell phone", "cellphone": "cell phone", "mug": "cup", "doughnut": "donut",
    "fridge": "refrigerator", "teddy": "teddy bear", "remote": "tv remote", "remote control": "tv remote",
    "keyboard": "computer keyboard", "hair dryer": "hair drier", "hairdryer": "hair drier",
    "table": "dining table", "plant": "potted plant", "ball": "sports ball", "football": "sports ball",
    "basketball": "sports ball", "soccer ball": "sports ball", "tennis ball": "sports ball",
}
IRREGULAR_PLURALS = {"person": ["people", "persons"], "man": ["men"], "woman": ["women"], "child": ["children"],
                     "kid": ["kids"], "knife": ["knives"], "sheep": ["sheep"], "skis": ["skis"],
                     "puppy": ["puppies"], "lady": ["ladies"]}

RELATIONS = [  # (phrase, canonical GenEval relation); longest phrases first when matching
    ("to the left of", "left of"), ("on the left of", "left of"), ("left of", "left of"),
    ("to the right of", "right of"), ("on the right of", "right of"), ("right of", "right of"),
    ("on top of", "above"), ("above", "above"), ("over", "above"),
    ("underneath", "below"), ("beneath", "below"), ("under", "below"), ("below", "below"),
]
SIDE_PHRASES = [("on the left", "left"), ("on the right", "right")]   # absolute: "the black cat on the left"
TEXT_WORDS = r"\b(says|saying|text|written|writing|word|words|letters|lettering|logo|sign that)\b"

# detector category names -> GenEval names
_TV_TO_GENEVAL = {"mouse": "computer mouse", "remote": "tv remote", "keyboard": "computer keyboard"}
_M2F_TO_GENEVAL = {"motorbike": "motorcycle", "aeroplane": "airplane", "sofa": "couch", "pottedplant": "potted plant",
                   "diningtable": "dining table", "tvmonitor": "tv", "mouse": "computer mouse", "remote": "tv remote",
                   "keyboard": "computer keyboard"}
MASK2FORMER = "facebook/mask2former-swin-small-coco-instance"   # the detector family/size official GenEval uses


def _plurals(name: str) -> set[str]:
    out = {name + "es" if name.endswith("s") else name + "s"}           # GenEval's own (sometimes odd) plural
    if re.search(r"(ch|sh|x|s)$", name):
        out.add(name + "es")
    last = name.split()[-1]
    for p in IRREGULAR_PLURALS.get(last, []):
        out.add(" ".join(name.split()[:-1] + [p]))
    return out


def _build_phrases():
    table = {}
    for c in CLASSES:
        table[c] = (c, False)
        for p in _plurals(c):
            table.setdefault(p, (c, True))
    for word, c in SYNONYMS.items():
        table.setdefault(word, (c, False))
        for p in _plurals(word):
            table.setdefault(p, (c, True))
    return {tuple(k.split()): v for k, v in table.items()}


_PHRASES = _build_phrases()
_MAX_LEN = max(len(k) for k in _PHRASES)


def _norm_words(text: str) -> list[str]:
    text = text.lower().replace("-", " ")
    text = re.sub(r"[^a-z0-9, ]+", " ", text).replace(",", " , ")
    return text.split()


def _match_at(words, i, table, max_len):
    for n in range(max_len, 0, -1):
        key = tuple(words[i:i + n])
        if len(key) == n and key in table:
            return key, table[key]
    return None, None


# ----------------------------------------------------------------------------- 1. prompt -> checklist
def parse_prompt(text: str) -> dict | None:
    """Turn a prompt into a GenEval-style checklist:
    {"include": [{"class", "count", "color"?, "position": [relation, target_index]?, "ref"?}], "exclude": [...]}.
    Returns None when the prompt mentions none of the 80 checkable object types."""
    words = _norm_words(text)
    rel_table = {tuple(p.split()): r for p, r in RELATIONS}
    side_table = {tuple(p.split()): s for p, s in SIDE_PHRASES}
    mentions, relations, sides = [], [], []
    i = 0
    while i < len(words):
        key, rel = _match_at(words, i, rel_table, 4)
        if key:
            relations.append((i, rel, len(key)))
            i += len(key)
            continue
        key, side = _match_at(words, i, side_table, 3)
        if key and (i + len(key) >= len(words) or words[i + len(key)] != "of"):
            sides.append((i, side))
            i += len(key)
            continue
        key, hit = _match_at(words, i, _PHRASES, _MAX_LEN)
        if key:
            cls, plural = hit
            nxt, _ = _match_at(words, i + len(key), _PHRASES, _MAX_LEN)
            if not (cls == "orange" and len(key) == 1 and nxt):        # "an orange cat": colour, not the fruit
                mentions.append({"class": cls, "plural": plural, "start": i, "end": i + len(key)})
            i += len(key)
            continue
        i += 1
    if not mentions:
        return None

    # modifiers in the few words before each mention: count, colour, "the" (= refers back to an earlier mention)
    prev_end = 0
    for m in mentions:
        lo = max(prev_end, m["start"] - 4)
        window = words[lo:m["start"]]
        if "," in window:
            window = window[len(window) - window[::-1].index(","):]
        m["color"] = next((w for w in reversed(window) if w in COLOURS), None)
        num = next((NUMBER_WORDS[w] for w in reversed(window) if w in NUMBER_WORDS), None)
        if num is None:
            num = next((int(w) for w in reversed(window) if w.isdigit() and 0 < int(w) <= 20), None)
        if num is None and "pair" in window:
            num = 2
        m["number"] = num
        m["the"] = bool(window) and window[-1] == "the" or (len(window) > 1 and window[-2] == "the")
        prev_end = m["end"]

    include, exclude, seen = [], [], Counter()
    for m in mentions:
        ref = m["the"] and seen[m["class"]] > 0 and m["number"] is None and not m["plural"]
        if m["number"] and m["number"] >= 2:
            req = {"class": m["class"], "count": m["number"]}
            exclude.append({"class": m["class"], "count": m["number"] + 1})       # exact count, like GenEval
        elif m["plural"] and not m["number"]:
            req = {"class": m["class"], "count": 2}                                  # "apples" = at least two
        else:
            req = {"class": m["class"], "count": 1}
        if m["color"]:
            req["color"] = m["color"]
        if ref:
            req["ref"] = True
        m["req"] = req
        include.append(req)
        seen[m["class"]] += 1

    # enumerated colours after a group: "three dogs, one black, one white, and one brown"
    extra = []
    for k, m in enumerate(mentions):
        if not (m["plural"] or (m["number"] or 0) >= 2):
            continue
        stop = mentions[k + 1]["start"] if k + 1 < len(mentions) else len(words)
        tail = words[m["end"]:stop]
        cols = [tail[j + 1] for j in range(len(tail) - 1) if tail[j] == "one" and tail[j + 1] in COLOURS]
        if len(cols) >= 2:
            extra += [{"class": m["class"], "count": 1, "color": c, "ref": True} for c in cols]

    # relations between the nearest mention before and after the phrase: "A left of B"
    links = []
    for pos, rel, n in relations:
        before = [m for m in mentions if m["end"] <= pos]
        after = [m for m in mentions if m["start"] >= pos + n]
        if before and after:
            links.append((before[-1], rel, after[0]))
    # absolute sides: "the black cat on the left and the white cat on the right" -> black cat left of white cat
    side_of = {}
    for pos, side in sides:
        before = [m for m in mentions if m["end"] <= pos]
        if before:
            side_of[side] = before[-1]
    if "left" in side_of and "right" in side_of and side_of["left"] is not side_of["right"]:
        links.append((side_of["left"], "left of", side_of["right"]))

    # GenEval convention: the target must come before the subject; subject stores [relation, target_index]
    order = [m["req"] for m in mentions]
    for subj, rel, tgt in links:
        s, t = order.index(subj["req"]), order.index(tgt["req"])
        if s < t:
            order.insert(t, order.pop(s))
    for subj, rel, tgt in links:
        subj["req"]["position"] = [rel, order.index(tgt["req"])]
    spec = {"include": order + extra}
    if exclude:
        spec["exclude"] = exclude
    return spec


def prompt_features(text: str, spec: dict | None = None) -> dict:
    """Cheap numeric clues about how hard a prompt is (inputs for the router)."""
    spec = parse_prompt(text) if spec is None else spec
    inc = spec["include"] if spec else []
    return {
        "n_words": len(text.split()),
        "n_commas": text.count(","),
        "structured": int(spec is not None),
        "n_objects": len(inc),
        "n_classes": len({r["class"] for r in inc}),
        "total_count": sum(r["count"] for r in inc if not r.get("ref")),
        "has_exact_count": int(bool(spec and spec.get("exclude"))),
        "n_colours": sum("color" in r for r in inc),
        "has_position": int(any("position" in r for r in inc)),
        "has_text": int(bool(re.search(TEXT_WORDS, text.lower()))),
    }


def describe_checks(spec: dict | None, check: dict | None) -> list[dict]:
    """One row per requirement for the app: {"kind", "label", "ok", "detail"?}.
    kind is "object" | "count" | "colour" | "position"; label reads like "2 × dog", "red car", "dog left of cat"."""
    if not spec:
        return []
    include, exclude = spec.get("include", []), spec.get("exclude", [])
    check = check or {}
    items = check.get("items") or [None] * len(include)
    ex_items = check.get("exclude_items") or [None] * len(exclude)
    counted = {r["class"] for r in exclude}
    too_many = {r["class"]: it["reason"] for r, it in zip(exclude, ex_items) if it and not it["ok"]}
    rows = []
    for req, item in zip(include, items):
        c, n = req["class"], req["count"]
        noun = f"{req['color']} {c}" if "color" in req else c
        is_count = (n > 1 or c in counted) and not req.get("ref")
        label = f"{n} × {noun}" if is_count else noun
        if "position" in req:
            rel, target = req["position"]
            label, kind = f"{label} {rel} {include[target]['class']}", "position"
        elif is_count:
            kind = "count"
        elif "color" in req:
            kind = "colour"
        else:
            kind = "object"
        ok = item["ok"] if item else bool(check.get("passed", True))
        detail = item["reason"] if item and not item["ok"] else None
        if kind == "count" and c in too_many:
            ok, detail = False, detail or too_many[c]
        row = {"kind": kind, "label": label, "ok": bool(ok)}
        if detail:
            row["detail"] = detail
        rows.append(row)
    return rows


# ----------------------------------------------------------------------------- geometry (GenEval)
def relative_position(box_a, box_b, threshold: float = 0.1) -> set[str]:
    """Position of A relative to B (GenEval's rule): centre offset, minus a margin proportional to object size."""
    boxes = np.array([box_a, box_b], dtype=np.float64)[:, :4].reshape(2, 2, 2)
    center_a, center_b = boxes.mean(axis=-2)
    dim_a, dim_b = np.abs(np.diff(boxes, axis=-2))[..., 0, :]
    offset = center_a - center_b
    revised = np.maximum(np.abs(offset) - threshold * (dim_a + dim_b), 0) * np.sign(offset)
    if np.all(np.abs(revised) < 1e-3):
        return set()
    dx, dy = revised / np.linalg.norm(offset)
    rels = set()
    if dx < -0.5:
        rels.add("left of")
    if dx > 0.5:
        rels.add("right of")
    if dy < -0.5:
        rels.add("above")
    if dy > 0.5:
        rels.add("below")
    return rels


# ----------------------------------------------------------------------------- 2. the checker
class SemanticChecker:
    """Loads a COCO detector + CLIP once; then checks pictures against prompts.

    detector="maskrcnn"    torchvision Mask R-CNN v2: fast, used by the cascade's GATE
    detector="mask2former" Mask2Former Swin-S (what GenEval uses): used as the independent final JUDGE, so the
                           cascade is never scored by the same models that steered it."""

    def __init__(self, device: str = "cuda", clip_name: str = "openai/clip-vit-large-patch14",
                 detector: str = "maskrcnn", det_threshold: float = 0.3, count_threshold: float = 0.9,
                 position_threshold: float = 0.1, max_objects: int = 16, mask2former_size: int = 1024):
        import torch
        from transformers import CLIPModel, CLIPProcessor

        self.torch, self.device, self.backend = torch, device, detector
        self.det_threshold, self.count_threshold = det_threshold, count_threshold
        self.position_threshold, self.max_objects = position_threshold, max_objects
        if detector == "maskrcnn":
            from torchvision.models.detection import MaskRCNN_ResNet50_FPN_V2_Weights, maskrcnn_resnet50_fpn_v2
            weights = MaskRCNN_ResNet50_FPN_V2_Weights.DEFAULT
            self.detector = maskrcnn_resnet50_fpn_v2(weights=weights,
                                                     box_score_thresh=min(det_threshold, count_threshold))
            self.det_names = [_TV_TO_GENEVAL.get(n, n) for n in weights.meta["categories"]]
        elif detector == "mask2former":
            from transformers import AutoImageProcessor, Mask2FormerForUniversalSegmentation
            self.det_proc = AutoImageProcessor.from_pretrained(MASK2FORMER)
            # The checkpoint defaults to a tiny 384x384 input; GenEval's Mask2Former runs at ~1024 (colours and
            # small objects are much better there). Square resize keeps masks aligned with the post-processing.
            self.det_proc.size = {"height": mask2former_size, "width": mask2former_size}
            self.detector = Mask2FormerForUniversalSegmentation.from_pretrained(MASK2FORMER)
            id2label = self.detector.config.id2label
            self.det_names = [_M2F_TO_GENEVAL.get(id2label[i], id2label[i]) for i in range(len(id2label))]
        else:
            raise ValueError(f"unknown detector {detector!r}")
        self.detector.eval().to(device)
        dtype = torch.float16 if str(device).startswith("cuda") else torch.float32
        self.clip = CLIPModel.from_pretrained(clip_name, torch_dtype=dtype).eval().to(device)
        self.proc = CLIPProcessor.from_pretrained(clip_name)
        self.dtype = dtype
        self._colour_text = {}

    # -- CLIP --------------------------------------------------------------------------------------
    def _image_embeds(self, pil_images):
        torch = self.torch
        with torch.no_grad():
            px = self.proc(images=pil_images, return_tensors="pt")["pixel_values"].to(self.device, self.dtype)
            e = self.clip.get_image_features(pixel_values=px)
            e = e if isinstance(e, torch.Tensor) else e.pooler_output
            return e / e.norm(dim=-1, keepdim=True)

    def _text_embeds(self, texts):
        torch = self.torch
        with torch.no_grad():
            tok = self.proc(text=texts, return_tensors="pt", padding=True, truncation=True, max_length=77).to(self.device)
            e = self.clip.get_text_features(**tok)
            e = e if isinstance(e, torch.Tensor) else e.pooler_output
            return e / e.norm(dim=-1, keepdim=True)

    def clip_scores(self, images, prompts) -> list[float]:
        """CLIP image-text match x100 (same formula as notebook 01's CLIP score)."""
        from PIL import Image
        pil = [im if isinstance(im, Image.Image) else Image.fromarray(np.asarray(im)) for im in images]
        sims = (self._image_embeds(pil) * self._text_embeds(list(prompts))).sum(-1)
        return (100 * sims).clamp(min=0).float().cpu().tolist()

    def colours(self, image, objects, classname) -> list[str]:
        """GenEval's colour test: CLIP zero-shot over 10 colours on each object's masked crop (grey background)."""
        from PIL import Image
        if classname not in self._colour_text:
            templates = ["a photo of a {c} {o}", "a photo of a {c}-colored {o}", "a photo of a {c} object"]
            emb = []
            for c in COLOURS:
                e = self._text_embeds([t.format(c=c, o=classname) for t in templates]).mean(0)
                emb.append(e / e.norm())
            self._colour_text[classname] = self.torch.stack(emb)
        rgb = np.asarray(image.convert("RGB") if isinstance(image, Image.Image) else image)
        crops = []
        for obj in objects:
            x1, y1, x2, y2 = [int(round(v)) for v in obj["box"]]
            img = rgb.copy()
            if obj.get("mask") is not None:
                img[~obj["mask"]] = 153                     # GenEval's #999 background
            crop = img[max(y1, 0):max(y2, y1 + 1), max(x1, 0):max(x2, x1 + 1)]
            crops.append(Image.fromarray(crop if crop.size else img))
        logits = self._image_embeds(crops) @ self._colour_text[classname].T
        return [COLOURS[i] for i in logits.argmax(-1).cpu().tolist()]

    # -- detector -----------------------------------------------------------------------------------
    def detect(self, image) -> dict:
        """COCO instance detection. Returns {class: [{"box", "mask", "score"}]}, most confident first."""
        torch = self.torch
        rgb = np.asarray(image.convert("RGB") if hasattr(image, "convert") else image)
        found = {}
        if self.backend == "maskrcnn":
            x = torch.from_numpy(rgb.copy()).permute(2, 0, 1).float().div(255).to(self.device)
            with torch.no_grad():
                out = self.detector([x])[0]
            boxes, labels, scores = out["boxes"].cpu().numpy(), out["labels"].cpu().numpy(), out["scores"].cpu().numpy()
            masks = (out["masks"][:, 0] > 0.5).cpu().numpy()
            items = [(self.det_names[int(lab)], b.tolist(), m, float(s)) for b, lab, s, m in zip(boxes, labels, scores, masks)]
        else:
            from PIL import Image
            inputs = self.det_proc(images=Image.fromarray(rgb), return_tensors="pt").to(self.device)
            with torch.no_grad():
                out = self.detector(**inputs)
            res = self.det_proc.post_process_instance_segmentation(
                out, threshold=min(self.det_threshold, self.count_threshold),
                target_sizes=[rgb.shape[:2]], return_binary_maps=True)[0]
            items = []
            for k, seg in enumerate(res["segments_info"]):
                m = res["segmentation"][k].bool().cpu().numpy()
                ys, xs = np.nonzero(m)
                if len(xs):
                    box = [float(xs.min()), float(ys.min()), float(xs.max() + 1), float(ys.max() + 1)]
                    items.append((self.det_names[seg["label_id"]], box, m, float(seg["score"])))
        for name, box, m, s in items:
            if name in CLASSES:
                found.setdefault(name, []).append({"box": box, "mask": m, "score": s})
        for name in found:
            found[name] = sorted(found[name], key=lambda d: -d["score"])[:self.max_objects]
        return found

    # -- structured check ---------------------------------------------------------------------------
    def check(self, image, spec: dict, detections: dict | None = None) -> dict:
        """GenEval-style: every 'include' must be satisfied (count, colour, position); every 'exclude' must not.
        Extension over GenEval: when one class appears in several requirements ("the black cat ... the white
        cat"), each requirement is matched to a different detected object."""
        t0 = time.perf_counter()
        dets = self.detect(image) if detections is None else detections
        # GenEval uses the strict counting threshold for counted objects; everything else uses the normal one
        counted = {r["class"] for r in spec.get("exclude", [])}
        objs = {c: [d for d in v if d["score"] >= (self.count_threshold if c in counted else self.det_threshold)]
                for c, v in dets.items()}
        include = spec.get("include", [])
        n_reqs = Counter(r["class"] for r in include)
        used, used_ref = {}, {}
        matched, reasons, ok = [], [], True
        items, exclude_items = [], []          # one verdict per requirement (for the app's checklist)
        for req in include:
            n_reasons = len(reasons)
            c, n = req["class"], req["count"]
            pool = objs.get(c, [])
            taken = used_ref.setdefault(c, set()) if req.get("ref") else used.setdefault(c, set())
            cand = [i for i in range(len(pool)) if i not in taken]
            if n_reqs[c] == 1 and not req.get("ref"):
                cand = cand[:n]                                  # GenEval: the n most confident objects
            chosen = None
            if len(cand) < n:
                reasons.append(f"expected {n} {c}, found {len(cand)}")
            elif "color" in req:
                cols = self.colours(image, [pool[i] for i in cand], c)
                hits = [i for i, col in zip(cand, cols) if col == req["color"]]
                if len(hits) >= n:
                    chosen = hits[:n]
                else:
                    reasons.append(f"expected {n} {req['color']} {c}, saw colours {dict(Counter(cols))}")
            else:
                chosen = cand[:n]
            if chosen is not None and "position" in req:
                rel, target = req["position"]
                tgt = matched[target] if target < len(matched) else None
                if tgt is None:
                    reasons.append(f"no {include[target]['class']} to be {rel}")
                    chosen = None
                else:
                    for a in chosen:
                        for b in tgt:
                            rels = relative_position(pool[a]["box"], b["box"], self.position_threshold)
                            if rel not in rels:
                                reasons.append(f"expected {c} {rel} {include[target]['class']}, "
                                               f"found {' and '.join(sorted(rels)) or 'overlapping'}")
                                chosen = None
                                break
                        if chosen is None:
                            break
            if chosen is None:
                ok = False
                matched.append(None)
            else:
                taken.update(chosen)
                matched.append([pool[i] for i in chosen])
            items.append({"ok": chosen is not None, "reason": "; ".join(reasons[n_reasons:]) or None})
        for req in spec.get("exclude", []):
            found = len(objs.get(req["class"], []))
            if found >= req["count"]:
                ok = False
                reasons.append(f"expected fewer than {req['count']} {req['class']}, found {found}")
                exclude_items.append({"ok": False, "reason": reasons[-1]})
            else:
                exclude_items.append({"ok": True, "reason": None})
        return {"passed": ok, "reasons": reasons, "n_detected": sum(len(v) for v in objs.values()),
                "items": items, "exclude_items": exclude_items, "ms": (time.perf_counter() - t0) * 1000}

    # -- the gate used by the cascade ---------------------------------------------------------------
    def gate(self, image, prompt: str, clip_min: float | None = None, spec: dict | None = None) -> dict:
        """Semantic verdict: structured check (if the prompt has checkable objects) AND CLIP >= clip_min."""
        t0 = time.perf_counter()
        spec = parse_prompt(prompt) if spec is None else spec
        clip = self.clip_scores([image], [prompt])[0]
        res = {"clip": clip, "structured": spec is not None, "check": None, "reasons": []}
        passed = True
        if clip_min is not None and clip < clip_min:
            passed = False
            res["reasons"].append(f"CLIP {clip:.1f} < {clip_min:.1f}")
        if spec is not None:
            res["check"] = self.check(image, spec)
            passed = passed and res["check"]["passed"]
            res["reasons"] += res["check"]["reasons"]
        res["passed"] = passed
        res["ms"] = (time.perf_counter() - t0) * 1000
        return res

In [ ]:
%%writefile p3_engine.py
"""P3 engine: the generator + the quality-gated cascade, in one place.

Used by notebook 03a (to MEASURE the low-memory setup) and by the Space backend (to SERVE it), so the numbers
in the write-up come from exactly the code that runs in the app.

    SETUPS / load_generator()  SANA-Sprint 0.6B in float16, or notebook 01's low-memory setups (M2, M3)
    Router                     02b's prompt -> starting-level model, run from its JSON export (numpy only)
    Engine.verdict()           the gate: visual check (classical CV) + semantic check (detector + CLIP)
    Engine.run()               one request: router -> draw -> check -> retry -> polish -> WebP -> API dict
"""

from __future__ import annotations

import base64
import gc
import os
import random
import time
import types
import zlib
from pathlib import Path

import numpy as np
import torch

import p3_classical as pcv
import p3_semantic as psem

MODEL_REPO = "Efficient-Large-Model/Sana_Sprint_0.6B_1024px_diffusers"

# Notebook 01's setups. "te" = the text reader (Gemma-2 2B), "tr" = the drawer (the diffusion transformer).
SETUPS = {
    "fp16": {},
    "M2": {"te_quant": "4bit", "tr_quant": "8bit", "te_embed_cpu": True, "vae_tiling": True},
    "M3": {"te_quant": "4bit", "tr_quant": "4bit", "te_embed_cpu": True, "vae_tiling": True},
}
SETUP_NAMES = {
    "fp16": "SANA-Sprint 0.6B · float16",
    "M2": "SANA-Sprint 0.6B · low-memory (4-bit reader, 8-bit drawer)",
    "M3": "SANA-Sprint 0.6B · low-memory (4-bit reader, 4-bit drawer)",
}

# What each app mode does. Every mode is one of the strategies replayed in notebooks 02b/03a, so its
# speed and accuracy are measured, not guessed. budget_s: never START another try that would end past this.
# start: a level number, or "router" to let 02b's router pick. 02b found the router no better than always
# starting cheap (46.5% vs 58% majority baseline), so no mode uses it by default; 03a still reports it.
DEFAULT_MODES = {
    "fast": {"start": 0, "cascade": False, "budget_s": None},
    "balanced": {"start": 0, "cascade": True, "budget_s": 6.0},
    "quality": {"start": 1, "cascade": True, "budget_s": 9.0},
}
MODE_STRATEGY = {"fast": "fixed R0", "balanced": "cascade from R0", "quality": "cascade from R1"}

SEED_STRIDE = 100000  # seed of level i = base seed + i * SEED_STRIDE (same rule as notebook 02b)


def free_memory():
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    try:
        import ctypes

        ctypes.CDLL("libc.so.6").malloc_trim(0)
    except Exception:
        pass


# ----------------------------------------------------------------------------- 1. the generator
def _quant_config(setup: dict, dtype):
    """Which parts of the pipeline get quantized, and how (identical to notebook 01)."""
    from diffusers import BitsAndBytesConfig as DiffusersBnbConfig
    from diffusers import PipelineQuantizationConfig
    from transformers import BitsAndBytesConfig as TransformersBnbConfig

    qm = {}
    te, tr = setup.get("te_quant"), setup.get("tr_quant")
    if te == "8bit":
        qm["text_encoder"] = TransformersBnbConfig(load_in_8bit=True)
    elif te == "4bit":
        qm["text_encoder"] = TransformersBnbConfig(load_in_4bit=True, bnb_4bit_quant_type="nf4",
                                                   bnb_4bit_use_double_quant=True, bnb_4bit_compute_dtype=dtype)
    if tr == "8bit":
        qm["transformer"] = DiffusersBnbConfig(load_in_8bit=True)
    elif tr == "4bit":
        qm["transformer"] = DiffusersBnbConfig(load_in_4bit=True, bnb_4bit_quant_type="nf4",
                                               bnb_4bit_compute_dtype=dtype)
    return PipelineQuantizationConfig(quant_mapping=qm) if qm else None


class CPUEmbedding(torch.nn.Module):
    """The reader's word table (256k words, 1.18 GB) kept on the CPU.
    Keeps Gemma's built-in x48 scale: dropping it breaks prompt understanding (CLIP fell from 27 to 9)."""

    def __init__(self, emb):
        super().__init__()
        self.weight_cpu = emb.weight.detach().to("cpu")
        self.num_embeddings, self.embedding_dim = self.weight_cpu.shape
        scale = getattr(emb, "embed_scale", None)
        self.scale = float(scale) if scale is not None else 1.0

    def forward(self, input_ids):
        out = torch.nn.functional.embedding(input_ids.to("cpu"), self.weight_cpu)
        return (out * self.scale).to(input_ids.device)


def move_word_table_to_cpu(pipe, log=print):
    te = pipe.text_encoder
    old = te.embed_tokens
    test_ids = torch.tensor([[2, 651, 4401, 235265]], device=old.weight.device)
    ref = old(test_ids).float().cpu()
    te.embed_tokens = CPUEmbedding(old)
    diff = (ref - te.embed_tokens(test_ids).float().cpu()).abs().max().item()
    log(f"  word table moved to CPU (scale x{te.embed_tokens.scale:.1f}), max difference vs original: {diff:.6f}")
    assert diff < 1e-3, "CPU word table doesn't match the original: stop, the pictures would be wrong"
    del old
    free_memory()


class MockPipe:
    """Offline testing only (no GPU, no model download). With P3_MOCK_IMAGES set: returns those photos
    (the cheapest level is sometimes blurred, like notebook 02b's mock). Otherwise: random sharp shapes."""

    def __init__(self, image_dir: str | None = None):
        self.files = sorted(Path(image_dir).glob("*.jpg")) if image_dir else []

    def __call__(self, prompt, generator, height, width, num_inference_steps, **kw):
        from PIL import Image, ImageDraw, ImageFilter

        images = []
        for p, g in zip(prompt, generator):
            k = zlib.crc32(p.encode()) + int(g.initial_seed())
            if self.files:
                im = Image.open(self.files[k % len(self.files)]).convert("RGB").resize((width, height))
                if width < 1024 and k % 3 == 0:
                    im = im.filter(ImageFilter.GaussianBlur(3))
            else:
                r = random.Random(k)
                im = Image.new("RGB", (width, height), tuple(r.randrange(40, 200) for _ in range(3)))
                d = ImageDraw.Draw(im)
                for _ in range(40):
                    x, y = r.randrange(width), r.randrange(height)
                    s = r.randrange(width // 20, width // 4)
                    shape = d.ellipse if r.random() < 0.5 else d.rectangle
                    shape([x, y, x + s, y + s], fill=tuple(r.randrange(256) for _ in range(3)))
            images.append(im)
        return types.SimpleNamespace(images=images)

    def set_progress_bar_config(self, **kw):
        pass


def load_generator(setup: str = "M3", device: str = "cuda", dtype=torch.float16, repo: str = MODEL_REPO,
                   mock: bool = False, log=print):
    """Load SANA-Sprint once. setup = "fp16" | "M2" | "M3" (see SETUPS)."""
    if mock:
        return MockPipe(os.environ.get("P3_MOCK_IMAGES"))
    from diffusers import SanaSprintPipeline

    cfg = SETUPS[setup]
    kwargs = {"torch_dtype": dtype}
    qc = _quant_config(cfg, dtype)
    if qc is not None:
        kwargs["quantization_config"] = qc
    try:
        pipe = SanaSprintPipeline.from_pretrained(repo, device_map=device, **kwargs)  # straight to GPU, saves RAM
    except Exception as e:
        log("device_map not supported here, loading the normal way:", str(e)[:120])
        free_memory()
        pipe = SanaSprintPipeline.from_pretrained(repo, **kwargs)
        pipe.to(device)
    if cfg.get("te_embed_cpu"):
        move_word_table_to_cpu(pipe, log)
    if cfg.get("vae_tiling"):
        pipe.vae.enable_tiling()  # decode in tiles: removes the memory spike at the end
    pipe.set_progress_bar_config(disable=True)
    return pipe


def model_size_mb(pipe) -> float | None:
    """All weights, like notebook 01's 'model size' (None for the mock)."""
    total = 0
    for name in ("text_encoder", "transformer", "vae"):
        m = getattr(pipe, name, None)
        if m is None:
            return None
        total += m.get_memory_footprint() if hasattr(m, "get_memory_footprint") else sum(
            p.numel() * p.element_size() for p in m.parameters())
    return total / 2**20


def call_kwargs(level: dict) -> dict:
    kw = {"height": level["res"], "width": level["res"], "num_inference_steps": level["steps"],
          "intermediate_timesteps": 1.3 if level["steps"] == 2 else None}
    if level["res"] != 1024:
        kw["use_resolution_binning"] = False  # otherwise it secretly renders 1024px and shrinks it
    return kw


@torch.inference_mode()
def generate_images(pipe, level: dict, prompts: list[str], seeds: list[int]):
    gens = [torch.Generator("cpu").manual_seed(int(s)) for s in seeds]
    return pipe(prompt=list(prompts), generator=gens, **call_kwargs(level)).images


def level_seed(base_seed: int, level: int) -> int:
    return int(base_seed + SEED_STRIDE * level) % 2**31


# ----------------------------------------------------------------------------- 2. the router
class Router:
    """Notebook 02b's logistic regression (prompt clues + PCA of the CLIP text embedding), from its JSON."""

    def __init__(self, export: dict | None):
        self.j = export or {}
        self.classes = [int(c) for c in self.j.get("classes", [])]
        if self.j.get("coef") is not None:
            self.mean = np.asarray(self.j["scaler_mean"], float)
            self.scale = np.asarray(self.j["scaler_scale"], float)
            self.pca_mean = np.asarray(self.j["pca_mean"], float)
            self.pca = np.asarray(self.j["pca_components"], float)
            self.coef = np.asarray(self.j["coef"], float)
            self.intercept = np.asarray(self.j["intercept"], float)

    @property
    def text_encoder(self) -> str | None:
        return self.j.get("text_encoder")

    def predict(self, prompt: str, spec: dict | None, embed_fn) -> tuple[int, float | None]:
        """-> (starting level, confidence 0-1 or None). embed_fn(list[str]) -> array (n, d)."""
        if not self.classes:
            return 0, None
        if self.j.get("coef") is None:  # every training label was the same: nothing was learned
            return self.classes[0], None
        feats = psem.prompt_features(prompt, spec)
        x_hand = (np.array([feats[f] for f in self.j["features"]], float) - self.mean) / self.scale
        e = np.asarray(embed_fn([prompt]), dtype=np.float64).reshape(-1)
        x = np.concatenate([x_hand, (e - self.pca_mean) @ self.pca.T])
        z = self.coef @ x + self.intercept
        if self.coef.shape[0] == 1:  # binary: sklearn predicts classes[1] when the decision value > 0
            p1 = 1.0 / (1.0 + np.exp(-z[0]))
            return self.classes[int(p1 > 0.5)], float(max(p1, 1 - p1))
        p = np.exp(z - z.max())
        p /= p.sum()
        i = int(p.argmax())
        return self.classes[i], float(p[i])


# ----------------------------------------------------------------------------- 3. the engine
class Engine:
    """Everything loaded once; then run() serves one request."""

    def __init__(self, config: dict, setup: str | None = None, device: str | None = None, mock: bool = False,
                 log=print):
        self.cfg, self.log, self.mock = config, log, mock
        self.device = device or ("cuda" if torch.cuda.is_available() else "cpu")
        on_gpu = str(self.device).startswith("cuda")
        if on_gpu:
            self.dtype = torch.bfloat16 if torch.cuda.get_device_capability(0)[0] >= 8 else torch.float16
        else:
            self.dtype = torch.float32
        self.levels = config["rungs"]
        self.top = len(self.levels) - 1
        gen = config.get("generator") or {}
        self.setup = setup or gen.get("setup") or "fp16"
        self.modes = {m: {**v, **{k: config.get("modes", {}).get(m, {}).get(k, v[k]) for k in v}}
                      for m, v in DEFAULT_MODES.items()}
        g = config["gate"]
        self.clip_min = float(g["clip_min"])
        self.unfixable = set(g.get("visual_unfixable_flags", pcv.UNFIXABLE))
        lat = config.get("rung_latency_s", {})
        gate_s = float(config.get("gate_latency_s", 0.3))
        self.expected_s = [float(lat.get(lv["name"], 1.5)) + gate_s for lv in self.levels]

        clip_name = os.environ.get("P3_GATE_CLIP", g["clip_model"])
        log(f"Loading the checker (detector {g.get('detector', 'maskrcnn')}, CLIP {clip_name})...")
        self.gate = psem.SemanticChecker(device=self.device, clip_name=clip_name,
                                         detector=g.get("detector", "maskrcnn"),
                                         det_threshold=g.get("det_threshold", 0.3),
                                         count_threshold=g.get("count_threshold", 0.9))
        self.router = Router(config.get("router"))
        if self.router.text_encoder and self.router.text_encoder != g["clip_model"]:
            log(f"WARNING: router was trained on {self.router.text_encoder} text features, gate uses {g['clip_model']}")
        log(f"Loading the generator ({'mock' if mock else self.setup})...")
        self.pipe = load_generator(self.setup, self.device, self.dtype, config.get("model", MODEL_REPO),
                                   mock=mock, log=log)
        free_memory()
        self.model_name = ("MOCK · " if mock else "") + SETUP_NAMES.get(self.setup, self.setup)
        self.gpu_name = torch.cuda.get_device_name(0) if on_gpu else "CPU"

    # -- pieces ----------------------------------------------------------------------------------
    def embed(self, texts):
        return self.gate._text_embeds(list(texts)).float().cpu().numpy()

    def draw(self, level: int, prompts: list[str], seeds: list[int]):
        return generate_images(self.pipe, self.levels[level], prompts, seeds)

    def verdict(self, image, prompt: str, spec: dict | None) -> dict:
        """The gate (same rule as notebook 02b): visual check has no unfixable flag AND the structured check
        passes (when the prompt has one) AND gate-CLIP >= clip_min."""
        t0 = time.perf_counter()
        vis = pcv.assess(np.asarray(image), prompt)
        bad = [f for f in vis["flags"] if f in self.unfixable]
        g = self.gate.gate(image, prompt, spec=spec)
        struct_ok = g["check"] is None or g["check"]["passed"]
        clip_ok = g["clip"] >= self.clip_min
        reasons = [f"picture looks {f.replace('_', ' ')}" for f in bad]
        if not clip_ok:
            reasons.append(f"weak match with the prompt (CLIP {g['clip']:.1f} < {self.clip_min:.1f})")
        if g["check"] is not None:
            reasons += g["check"]["reasons"]
        return {"passed": not bad and struct_ok and clip_ok, "vis": vis, "vis_ok": not bad, "gate": g,
                "struct_ok": struct_ok, "clip": float(g["clip"]), "reasons": reasons,
                "seconds": time.perf_counter() - t0}

    def start_level(self, mode: str, prompt: str, spec: dict | None) -> tuple[int, float | None]:
        start = self.modes[mode]["start"]
        if start == "router":
            level, conf = self.router.predict(prompt, spec, self.embed)
        else:
            level, conf = int(start), None
        return min(max(int(level), 0), self.top), conf

    def warm_up(self):
        """First GPU calls are slow (kernel set-up): run every level once before real users arrive."""
        for i in range(len(self.levels)):
            im = self.draw(i, ["a photo of a cat"], [0])[0]
        self.verdict(im, "a photo of a cat", psem.parse_prompt("a photo of a cat"))

    # -- one request ---------------------------------------------------------------------------------
    def run(self, prompt: str, mode: str = "balanced", seed: int | None = None) -> dict:
        """Router -> draw -> check -> (retry at the next level with a new seed) -> polish -> WebP.
        Returns the API response (docs/API_CONTRACT.md in the frontend)."""
        t0 = time.perf_counter()
        policy = self.modes[mode]
        spec = psem.parse_prompt(prompt)
        start, conf = self.start_level(mode, prompt, spec)
        base_seed = int(seed) if seed is not None else random.randrange(2**31)
        attempts, gen_s, check_s = [], 0.0, 0.0
        level = start
        while True:
            s = level_seed(base_seed, level)
            tg = time.perf_counter()
            image = self.draw(level, [prompt], [s])[0]
            if self.device.startswith("cuda"):
                torch.cuda.synchronize()
            g_time = time.perf_counter() - tg
            v = self.verdict(image, prompt, spec)
            gen_s += g_time
            check_s += v["seconds"]
            attempts.append({"level": level, "resolution": self.levels[level]["res"], "seed": s,
                             "time_s": round(g_time + v["seconds"], 3), "passed": bool(v["passed"]),
                             "reasons": [] if v["passed"] else (v["reasons"] or ["did not pass the check"])})
            if v["passed"] or not policy["cascade"] or level >= self.top:
                break
            budget = policy.get("budget_s")
            if budget and (time.perf_counter() - t0) + self.expected_s[level + 1] > budget:
                break
            level += 1

        tp = time.perf_counter()
        polished, pinfo = pcv.safe_enhance(np.asarray(image), prompt)
        polish_s = time.perf_counter() - tp
        te = time.perf_counter()
        data, pk = pcv.pack_for_mobile(polished, max_side=1024)
        encode_s = time.perf_counter() - te
        h, w = np.asarray(polished).shape[:2]
        return {
            "image": "data:image/webp;base64," + base64.b64encode(data).decode("ascii"),
            "width": int(w),
            "height": int(h),
            "prompt": prompt,
            "mode": mode,
            "passed": bool(v["passed"]),
            "level_used": level,
            "levels_total": len(self.levels),
            "attempts": attempts,
            "timing": {"total_s": round(time.perf_counter() - t0, 3), "generate_s": round(gen_s, 3),
                       "check_s": round(check_s, 3), "polish_s": round(polish_s, 3), "encode_s": round(encode_s, 3)},
            "checks": {
                "clip_score": round(v["clip"], 1),
                "visual": {"ok": bool(v["vis_ok"]), "flags": list(v["vis"]["flags"])},
                "semantic": psem.describe_checks(spec, v["gate"]["check"]),
            },
            "router": {"start_level": start, "confidence": None if conf is None else round(conf, 3)},
            "polish": {"applied": bool(pinfo["accepted"]), "steps": list(pinfo["actions"]) if pinfo["accepted"] else []},
            "model": self.model_name,
            "gpu": self.gpu_name,
            **({"mock": True} if self.mock else {}),
        }

In [ ]:
import json, time, shutil, importlib, textwrap, zipfile
from pathlib import Path
import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt
from PIL import Image
from IPython.display import display
try:
    from tqdm.auto import tqdm
except ImportError:
    tqdm = lambda x, **k: x
import p3_classical as pcv
import p3_semantic as psem
import p3_engine as pe
for _m in (pcv, psem, pe):
    importlib.reload(_m)

MOCK = os.environ.get("P3_MOCK_GENERATOR") == "1"     # offline testing only: fake generator, no GPU
DEV = "cuda" if torch.cuda.is_available() else "cpu"
assert DEV == "cuda" or MOCK, "No GPU found: Runtime > Change runtime type > T4 GPU, then Run all."
GEN_BS = 2
JUDGE_CLIP = os.environ.get("P3_JUDGE_CLIP", JUDGE_CLIP)
print("Device:", torch.cuda.get_device_name(0) if DEV == "cuda" else "CPU (mock mode)")

def sync_time():
    if DEV == "cuda":
        torch.cuda.synchronize()
    return time.perf_counter()

# Kaggle: no Drive, so get the repo (it holds 02b's results in results/02b) and work in /kaggle/working.
KAGGLE = Path(os.environ.get("P3_KAGGLE_WORKING", "/kaggle/working"))
ON_KAGGLE = KAGGLE.exists() and "P3_02B_OUT" not in os.environ
if ON_KAGGLE:
    import subprocess
    REPO = KAGGLE / "text-image-CV"
    if not REPO.exists():
        subprocess.run(["git", "clone", "--depth", "1", f"https://github.com/{GITHUB_REPO}.git", str(REPO)], check=True)
    os.environ.update(P3_02B_OUT=str(REPO / "results" / "02b"), P3_OUT=str(KAGGLE / "03a_out"),
                      P3_WORK=str(KAGGLE / "03a_work"))
    SAVE_TO_DRIVE = False
    print("Kaggle: using", REPO, "(GitHub", GITHUB_REPO + ")")
IN_02B = Path(os.environ.get("P3_02B_OUT", "/content/P3_02b_out"))
SHARED = IN_02B / "shared"
OUT = Path(os.environ.get("P3_OUT", "/content/P3_03a_out"))
WORK = Path(os.environ.get("P3_WORK", "/content/p3_03a_work"))
if SAVE_TO_DRIVE:
    try:
        from google.colab import drive
        drive.mount("/content/drive")
        IN_02B = Path("/content/drive/MyDrive/P3_T2I/02b_router_cascade")
        SHARED = Path("/content/drive/MyDrive/P3_T2I/shared")
        OUT = Path("/content/drive/MyDrive/P3_T2I/03a_lowmem_check")
    except Exception as e:
        print("Google Drive not available, using local folders:", e)
if MODE_TAG is None:
    MODE_TAG = "full" if (IN_02B / "matrix_full.csv").exists() and (IN_02B / "prompts_full.json").exists() else "quick"
TAG = f"{LOWMEM_SETUP}_{MODE_TAG}"
FIG = OUT / f"figures_{TAG}"
IMG_DIR = WORK / f"images_{TAG}"
for d in (OUT, WORK, FIG, IMG_DIR):
    d.mkdir(parents=True, exist_ok=True)
print("Reading 02b from:", IN_02B, "| mode:", MODE_TAG, "| writing to:", OUT)

## 4 · Load 02b's results (test prompts, results per picture, router, gate settings)

In [ ]:
NEED = {"prompts": IN_02B / f"prompts_{MODE_TAG}.json", "matrix": IN_02B / f"matrix_{MODE_TAG}.csv",
        "config": SHARED / "cascade_config.json"}
missing = [str(p) for p in NEED.values() if not p.exists()]
assert not missing, "Notebook 02b hasn't finished (or saved elsewhere). Missing:\n  " + "\n  ".join(missing)

PROMPTS = json.load(open(NEED["prompts"], encoding="utf-8"))
CONFIG = json.load(open(NEED["config"], encoding="utf-8"))
if CONFIG.get("mode") != MODE_TAG:
    print(f"NOTE: cascade_config.json says mode={CONFIG.get('mode')!r}, using {MODE_TAG!r} prompts/matrix")
RUNGS = CONFIG["rungs"]
TOP = len(RUNGS) - 1
CLIP_MIN = float(CONFIG["gate"]["clip_min"])
LAT_16 = [float(CONFIG["rung_latency_s"][r["name"]]) for r in RUNGS]
GATE_16 = float(CONFIG["gate_latency_s"])
test_prompts = [p for p in PROMPTS if p["split"] == "test"]
if os.environ.get("P3_MAX_TEST"):                      # offline testing only: a few prompts
    test_prompts = test_prompts[:int(os.environ["P3_MAX_TEST"])]
TEST_PIDS = {p["pid"] for p in test_prompts}
BY_PID = {p["pid"]: p for p in PROMPTS}

COLS = ["pid", "split", "source", "tag", "rung", "rung_idx", "seed", "vis_flags", "vis_ok", "vis_ms",
        "gate_clip", "gate_struct", "gate_ms", "gate_reasons", "judge_clip", "judge_struct", "judge_reasons"]

def read_csv_tolerant(path, cols=COLS):
    """Read a matrix CSV even if an old header has fewer columns than the rows (02b's early bug)."""
    import csv
    with open(path, newline="", encoding="utf-8") as f:
        rows = list(csv.reader(f))
    header, body = rows[0], rows[1:]
    if header != cols[:len(header)] or len(header) == len(cols):
        return pd.read_csv(path)
    body = [r + [""] * (len(cols) - len(r)) for r in body if len(r) in (len(header), len(cols))]
    df = pd.DataFrame(body, columns=cols).replace("", np.nan)
    for c in ("pid", "rung_idx", "seed", "vis_ms", "gate_clip", "gate_ms", "judge_clip"):
        df[c] = pd.to_numeric(df[c], errors="coerce")
    return df

M16 = read_csv_tolerant(NEED["matrix"])
M16 = M16[M16.pid.isin(TEST_PIDS)].drop_duplicates(["pid", "rung"], keep="last")
TEACHER_CSV = IN_02B / f"teacher_{MODE_TAG}.csv"
TEACHER_LAT_FILE = IN_02B / f"teacher_latency_{MODE_TAG}.json"
S16_FILE = IN_02B / f"strategies_{MODE_TAG}.csv"
print(f"{len(test_prompts)} test prompts ({sum(p['source'] == 'geneval' for p in test_prompts)} GenEval-style, "
      f"{sum(p['source'] == 'coco' for p in test_prompts)} COCO) | 02b float16 pictures: {len(M16)}")
print(f"Gate: CLIP >= {CLIP_MIN:.2f} + visual + structured check | levels: "
      + ", ".join(f"{r['name']} {r['res']}px" for r in RUNGS))
print("Float16 seconds per level (02b):", dict(zip([r["name"] for r in RUNGS], np.round(LAT_16, 3))), "| gate:", round(GATE_16, 3))

## 5 · Load the app's setup (low-memory model + gate) and measure speed, memory and size
This is exactly what the app loads at start-up: `pe.Engine` = low-memory SANA-Sprint + the gate (Mask R-CNN + CLIP ViT-B/16) + the router.

In [ ]:
if DEV == "cuda":
    torch.cuda.empty_cache()
    torch.cuda.reset_peak_memory_stats()
engine = pe.Engine(CONFIG, setup=LOWMEM_SETUP, device=DEV, mock=MOCK)
SIZE_MB = pe.model_size_mb(engine.pipe)
LOADED_GB = torch.cuda.memory_allocated() / 1e9 if DEV == "cuda" else None

def time_level(i, n=8):
    for k in range(2):
        engine.draw(i, [test_prompts[k % len(test_prompts)]["prompt"]], [SEED])     # warm-up
    times = []
    for k in range(n):
        t0 = sync_time()
        engine.draw(i, [test_prompts[k % len(test_prompts)]["prompt"]], [SEED + k])
        times.append(sync_time() - t0)
    return float(np.median(times))

LAT_LOW = [time_level(i) for i in range(len(RUNGS))]
sample = engine.draw(TOP, [test_prompts[0]["prompt"]], [SEED])[0]
GATE_LOW = float(np.median([engine.verdict(sample, p["prompt"], p["spec"])["seconds"] for p in test_prompts[:6]]))

APP_PEAK_GB = None
if DEV == "cuda":
    torch.cuda.reset_peak_memory_stats()
    engine.run(test_prompts[0]["prompt"], "quality", seed=SEED)      # one full request at full size
    APP_PEAK_GB = torch.cuda.max_memory_allocated() / 1e9

speed = pd.DataFrame({
    "float16 (02b)": {**{f"draw {r['name']} ({r['res']}px)": v for r, v in zip(RUNGS, LAT_16)}, "gate check": GATE_16},
    f"low-memory {LOWMEM_SETUP}": {**{f"draw {r['name']} ({r['res']}px)": v for r, v in zip(RUNGS, LAT_LOW)}, "gate check": GATE_LOW},
})
display(speed.round(3))
speed.to_csv(OUT / f"speed_{TAG}.csv")
print(f"Model size (all weights): {SIZE_MB:.0f} MB" if SIZE_MB else "Model size: n/a (mock)")
if DEV == "cuda":
    print(f"GPU memory, everything loaded (generator + gate): {LOADED_GB:.2f} GB; "
          f"peak during a full 1024px request: {APP_PEAK_GB:.2f} GB (T4 has 15 GB)")

## 6 · Load the independent judge (Mask2Former + CLIP ViT-L/14, the same as 02b)
The judge is only for measuring: the app never loads it.

In [ ]:
judge = psem.SemanticChecker(device=DEV, clip_name=JUDGE_CLIP, detector="mask2former")
pe.free_memory()

## 7 · Every test prompt at every level with the low-memory model (resumable)
Same seeds as 02b (seed = 0 + 100000 × level + prompt id), so each picture is the low-memory twin of 02b's picture.

In [ ]:
LOW_CSV = OUT / f"matrix_{TAG}.csv"
done = set()
if LOW_CSV.exists():
    old = pd.read_csv(LOW_CSV)
    done = set(zip(old["pid"], old["rung"]))
    print(f"Resuming: {len(done)} pictures already done")
buffer = []

def flush():
    global buffer
    if buffer:
        pd.DataFrame(buffer).reindex(columns=COLS).to_csv(LOW_CSV, mode="a", header=not LOW_CSV.exists(), index=False)
        buffer = []

t_start = time.time()
for ri, rung in enumerate(RUNGS):
    todo = [p for p in test_prompts if (p["pid"], rung["name"]) not in done]
    for k in tqdm(range(0, len(todo), GEN_BS), desc=f"{rung['name']} ({rung['res']}px, {LOWMEM_SETUP})"):
        batch = todo[k:k + GEN_BS]
        seeds = [pe.level_seed(SEED + p["pid"], ri) for p in batch]
        images = engine.draw(ri, [p["prompt"] for p in batch], seeds)
        for p, im, seed in zip(batch, images, seeds):
            v = engine.verdict(im, p["prompt"], p["spec"])
            j = judge.gate(im, p["prompt"], spec=p["spec"])
            buffer.append(dict(
                pid=p["pid"], split=p["split"], source=p["source"], tag=p["tag"], rung=rung["name"], rung_idx=ri,
                seed=seed, vis_flags=",".join(v["vis"]["flags"]), vis_ok=v["vis_ok"], vis_ms=None,
                gate_clip=v["clip"], gate_struct=None if v["gate"]["check"] is None else v["gate"]["check"]["passed"],
                gate_ms=v["seconds"] * 1000, gate_reasons="; ".join(v["reasons"])[:300], judge_clip=j["clip"],
                judge_struct=None if j["check"] is None else j["check"]["passed"],
                judge_reasons="; ".join(j["reasons"])[:300]))
            small = im if im.width <= 768 else im.resize((768, 768), Image.LANCZOS)
            small.save(IMG_DIR / f"{p['pid']}_{rung['name']}.jpg", quality=92)
        if len(buffer) >= 20:
            flush()
    flush()
print(f"Finished in {(time.time() - t_start) / 60:.1f} min")

## 8 · Replay the strategies for BOTH models (scored by the independent judge)
Same replay as 02b section 11: latency = every level tried + the gate check after each try; quality = the judge's verdict on the picture finally returned. **First, a self-check:** replaying 02b's own float16 results with our exported router must reproduce 02b's table.

In [ ]:
def as_bool(s):
    return s.map({True: True, False: False, "True": True, "False": False, 1.0: True, 0.0: False})

def prep(M):
    M = M[M.pid.isin(TEST_PIDS)].drop_duplicates(["pid", "rung"], keep="last").copy()
    M["vis_ok"] = as_bool(M["vis_ok"]).astype(bool)
    M["gate_struct_ok"] = as_bool(M["gate_struct"]).fillna(True).astype(bool)
    M["gate_pass"] = M["vis_ok"] & M["gate_struct_ok"] & (M["gate_clip"] >= CLIP_MIN)
    M["judge_struct_ok"] = as_bool(M["judge_struct"])
    return M.set_index(["pid", "rung_idx"]).sort_index()

T16 = prep(M16)
TLOW = prep(pd.read_csv(LOW_CSV))
complete = [pid for pid in sorted(TEST_PIDS)
            if all((pid, i) in T16.index for i in range(len(RUNGS))) and all((pid, i) in TLOW.index for i in range(len(RUNGS)))]
print(f"Prompts with every level in both models: {len(complete)} of {len(TEST_PIDS)}")

ROUTER_START = {pid: engine.router.predict(BY_PID[pid]["prompt"], BY_PID[pid]["spec"], engine.embed)[0] for pid in complete}

def replay(T, lat, gate_lat, start_of, cascade, budget=None, always_check=False):
    out = []
    for pid in complete:
        r = int(start_of(pid))
        cost = 0.0
        while True:
            row = T.loc[(pid, r)]
            cost += lat[r]
            if cascade or always_check:
                cost += gate_lat
            if not cascade or row["gate_pass"] or r == TOP:
                break
            if budget and cost + lat[r + 1] + gate_lat > budget:
                break
            r += 1
        out.append(dict(pid=pid, final=r, latency=cost, source=row["source"], tag=row["tag"], gate_pass=bool(row["gate_pass"]),
                        judge_struct=row["judge_struct_ok"], judge_clip=row["judge_clip"], vis_ok=row["vis_ok"]))
    return pd.DataFrame(out)

STRATEGIES = {f"fixed {r['name']} ({r['steps']} step, {r['res']}px)": (lambda pid, i=i: i, False) for i, r in enumerate(RUNGS)}
STRATEGIES.update({
    "cascade from R0": (lambda pid: 0, True),
    "router only": (lambda pid: ROUTER_START[pid], False),
    "router + cascade": (lambda pid: ROUTER_START[pid], True),
    "cascade from R1": (lambda pid: min(1, TOP), True),
})

def summarize(results):
    rows = []
    for name, df in results.items():
        g, c = df[df.source == "geneval"], df[df.source == "coco"]
        row = {"strategy": name, "avg_latency_s": df.latency.mean(), "p90_latency_s": df.latency.quantile(0.9),
               "max_latency_s": df.latency.max(),
               "geneval_style_acc": g.judge_struct.astype(float).mean() if len(g) else np.nan,
               "clip_L_coco": c.judge_clip.mean() if len(c) else np.nan, "clip_L_all": df.judge_clip.mean(),
               "visual_ok": df.vis_ok.astype(float).mean()}
        for i, r in enumerate(RUNGS):
            row[f"ends_at_{r['name']}"] = (df.final == i).mean()
        rows.append(row)
    return pd.DataFrame(rows).set_index("strategy")

R16 = {n: replay(T16, LAT_16, GATE_16, s, c) for n, (s, c) in STRATEGIES.items()}
RLOW = {n: replay(TLOW, LAT_LOW, GATE_LOW, s, c) for n, (s, c) in STRATEGIES.items()}
S16, SLOW = summarize(R16), summarize(RLOW)

# Self-check against 02b's own table (only meaningful when every test prompt is complete)
if S16_FILE.exists() and len(complete) == len(TEST_PIDS) and not os.environ.get("P3_MAX_TEST"):
    old = pd.read_csv(S16_FILE).set_index("strategy")
    for group, names in (("fixed + cascade-from-R0", [s for s in S16.index if s in old.index and "router" not in s]),
                         ("router (exported JSON vs 02b's sklearn)", [s for s in S16.index if s in old.index and "router" in s])):
        if not names:
            continue
        diff = max(float(np.nanmax(np.abs(S16.loc[names, c] - old.loc[names, c]))) for c in ("geneval_style_acc", "clip_L_all", "avg_latency_s"))
        ok = diff < (1e-6 if "router" not in group else 0.02)    # router: fp16 text embeddings may flip a near-tie
        print(("✓ " if ok else "✗ ") + f"replay of 02b's float16 results vs 02b's own table, {group}: largest difference {diff:.3g}")
else:
    print("Self-check skipped (02b's strategies table missing or some prompts incomplete).")

TEACHER = None
if TEACHER_CSV.exists() and TEACHER_LAT_FILE.exists():
    Tt = pd.read_csv(TEACHER_CSV).drop_duplicates("pid", keep="last")
    Tt = Tt[Tt.pid.isin(complete)]
    TEACHER_LAT = json.load(open(TEACHER_LAT_FILE))["seconds"]
    TEACHER = pd.DataFrame(dict(pid=Tt.pid, final=-1, latency=TEACHER_LAT, source=Tt.source, tag=Tt.tag,
                                judge_struct=as_bool(Tt.judge_struct), judge_clip=Tt.judge_clip, vis_ok=as_bool(Tt.vis_ok)))
    S_T = summarize({"ORIGINAL teacher (20 steps, 1024px)": TEACHER})

cols = ["geneval_style_acc", "clip_L_all", "avg_latency_s", "p90_latency_s"]
SIDE = pd.concat({"float16 (02b)": S16[cols], f"low-memory {LOWMEM_SETUP}": SLOW[cols]}, axis=1)
SIDE[("change", "accuracy")] = SLOW["geneval_style_acc"] - S16["geneval_style_acc"]
SIDE[("change", "CLIP-L")] = SLOW["clip_L_all"] - S16["clip_L_all"]
if TEACHER is not None:
    print("ORIGINAL teacher on the same prompts:", S_T[cols].round(3).to_dict("records")[0])
display(SIDE.round(3))
SIDE.to_csv(OUT / f"fp16_vs_lowmem_{TAG}.csv")
SLOW.to_csv(OUT / f"strategies_{TAG}.csv")

MAIN = pe.MODE_STRATEGY["balanced"]          # the app's default mode = the headline strategy
ours16, oursL = S16.loc[MAIN], SLOW.loc[MAIN]
d_acc = oursL.geneval_style_acc - ours16.geneval_style_acc
n_g = sum(BY_PID[p]["source"] == "geneval" for p in complete)
print()
print(f"{MAIN} (the app's Balanced mode): GenEval-style accuracy {ours16.geneval_style_acc:.1%} (float16) -> {oursL.geneval_style_acc:.1%} "
      f"(low-memory), change {d_acc:+.1%} on {n_g} prompts (1 prompt = {1 / max(n_g, 1):.1%}).")
if abs(d_acc) <= 0.03:
    VERDICT = "Low-memory keeps the retry system's accuracy within 3 points: safe to deploy."
elif d_acc > 0:
    VERDICT = "Low-memory is even slightly better here (likely noise at this sample size): safe to deploy."
else:
    VERDICT = (f"Low-memory loses {-d_acc:.1%}: try LOWMEM_SETUP = 'M2' (8-bit drawer), or deploy float16 "
               "(still fits a T4 with the gate) and quote float16 numbers for the app.")
print(VERDICT)
if TEACHER is not None:
    t = S_T.iloc[0]
    print(f"Ours (low-memory, {MAIN}) vs ORIGINAL: accuracy {oursL.geneval_style_acc:.1%} vs {t.geneval_style_acc:.1%}, "
          f"CLIP-L {oursL.clip_L_all:.2f} vs {t.clip_L_all:.2f}, time {oursL.avg_latency_s:.2f} s vs {t.avg_latency_s:.2f} s "
          f"({t.avg_latency_s / oursL.avg_latency_s:.1f}x faster)")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
for ax, col, label in [(axes[0], "geneval_style_acc", "GenEval-style accuracy (judge)"), (axes[1], "clip_L_all", "CLIP ViT-L/14 (judge)")]:
    for S, marker, name in [(S16, "o", "float16 (02b)"), (SLOW, "^", f"low-memory {LOWMEM_SETUP}")]:
        ax.scatter(S["avg_latency_s"], S[col], s=70, marker=marker, label=name)
        for strat, rr in S.iterrows():
            ax.annotate(strat, (rr["avg_latency_s"], rr[col]), fontsize=7, xytext=(4, 3), textcoords="offset points")
    if TEACHER is not None:
        ax.scatter(S_T["avg_latency_s"], S_T[col], s=90, marker="*", color="k", label="ORIGINAL teacher")
    ax.set_xlabel("average seconds per prompt (generation + checks). Left = faster")
    ax.set_ylabel(label + ". Up = better")
    ax.grid(alpha=0.3)
    ax.legend(fontsize=8)
plt.suptitle("Same strategies, float16 vs low-memory model, held-out test prompts")
plt.tight_layout()
plt.savefig(FIG / "fp16_vs_lowmem.png", dpi=140)
plt.show()

## 9 · The app's three modes, measured
Each mode is one of the replayed strategies (`p3_engine.DEFAULT_MODES`): **Fast** = cheapest level only, **Balanced** = cascade from the cheapest level, **Quality** = cascade from full size. (02b showed the router doesn't beat always starting cheap, so no mode uses it; it stays in the table above for the ablation.) Here every try counts the check after it (the app always shows the check), with each mode's time budget. Then a few **live** requests through the full app pipeline (polish + WebP included) confirm the replay.

In [ ]:
def mode_start(m):
    s = pe.DEFAULT_MODES[m]["start"]
    return (lambda pid: ROUTER_START[pid]) if s == "router" else (lambda pid, s=int(s): min(s, TOP))
MODE_START = {m: mode_start(m) for m in pe.DEFAULT_MODES}
mode_rows, MODE_STATS = [], {}
for m, policy in pe.DEFAULT_MODES.items():
    df = replay(TLOW, LAT_LOW, GATE_LOW, MODE_START[m], policy["cascade"], budget=policy["budget_s"], always_check=True)
    g = df[df.source == "geneval"]
    stats = {"strategy": pe.MODE_STRATEGY[m], "avg_s": float(df.latency.mean()), "p90_s": float(df.latency.quantile(0.9)),
             "max_s": float(df.latency.max()), "geneval_style_acc": float(g.judge_struct.astype(float).mean()) if len(g) else None,
             "clip_L_all": float(df.judge_clip.mean()), "gate_pass_rate": float(df.gate_pass.mean()),
             "avg_tries": float((df.final - df.pid.map(lambda p: MODE_START[m](p)) + 1).mean())}
    MODE_STATS[m] = stats
    mode_rows.append({"mode": m, **stats})
MODES_TABLE = pd.DataFrame(mode_rows).set_index("mode")
display(MODES_TABLE.round(3))

live = []
for m in pe.DEFAULT_MODES:
    for p in test_prompts[:N_LIVE]:
        r = engine.run(p["prompt"], m, seed=SEED + p["pid"])
        live.append({"mode": m, "total_s": r["timing"]["total_s"], "tries": len(r["attempts"]), "passed": r["passed"],
                     "kb": len(r["image"]) * 3 / 4 / 1024})
LIVE = pd.DataFrame(live).groupby("mode").agg(avg_s=("total_s", "mean"), max_s=("total_s", "max"),
                                              avg_tries=("tries", "mean"), passed=("passed", "mean"), avg_kb=("kb", "mean"))
print(f"Live requests ({N_LIVE} per mode, full pipeline incl. polish + WebP, no network):")
display(LIVE.round(3))
for m in pe.DEFAULT_MODES:
    MODE_STATS[m]["live_avg_s"] = float(LIVE.loc[m, "avg_s"])
    MODE_STATS[m]["live_avg_kb"] = float(LIVE.loc[m, "avg_kb"])

## 10 · Same prompt, same seed: float16 vs low-memory (for the write-up)
Uses 02b's saved test pictures (`test_images_*.zip` in Drive) if they're there.

In [ ]:
zip16 = IN_02B / f"test_images_{MODE_TAG}.zip"
if zip16.exists():
    with zipfile.ZipFile(zip16) as z:
        names = set(z.namelist())
        picks = [pid for pid in complete if f"{pid}_{RUNGS[TOP]['name']}.jpg" in names][:4]
        if picks:
            fig, axes = plt.subplots(len(picks), 2, figsize=(7, 3.4 * len(picks)))
            axes = np.array(axes).reshape(len(picks), 2)
            for pid, row in zip(picks, axes):
                name = f"{pid}_{RUNGS[TOP]['name']}.jpg"
                with z.open(name) as f:
                    a = Image.open(f).convert("RGB")
                b = Image.open(IMG_DIR / name)
                for ax, im, t in zip(row, (a, b), ("float16 (~8 GB)", f"low-memory {LOWMEM_SETUP} (~3 GB)")):
                    ax.imshow(im)
                    ax.set_title(t, fontsize=8)
                    ax.axis("off")
                row[0].text(0, -40, textwrap.fill(BY_PID[pid]["prompt"], 60), fontsize=7)
            plt.tight_layout()
            plt.savefig(FIG / "same_seed_fp16_vs_lowmem.png", dpi=110)
            plt.show()
else:
    print("02b's picture zip not found: skipping the visual comparison.")

## 11 · Save the app's configuration and a summary

In [ ]:
LOW_CONFIG = dict(CONFIG)
LOW_CONFIG.update({
    "generator": {"setup": LOWMEM_SETUP, "name": pe.SETUP_NAMES[LOWMEM_SETUP], "model_size_mb": SIZE_MB,
                  "gpu_memory_loaded_gb": LOADED_GB, "gpu_memory_peak_gb": APP_PEAK_GB},
    "rung_latency_s": {r["name"]: v for r, v in zip(RUNGS, LAT_LOW)},
    "gate_latency_s": GATE_LOW,
    "modes": {m: {**pe.DEFAULT_MODES[m], "measured": MODE_STATS[m]} for m in pe.DEFAULT_MODES},
    "measured_on": torch.cuda.get_device_name(0) if DEV == "cuda" else "CPU (mock)",
    "source": f"notebook 03a ({MODE_TAG}, {len(complete)} test prompts)",
    "verdict": VERDICT,
})
json.dump(LOW_CONFIG, open(SHARED / "cascade_config_lowmem.json", "w"), indent=1)
shutil.copy(SHARED / "cascade_config_lowmem.json", OUT / "cascade_config_lowmem.json")   # also next to the results
shutil.make_archive(str(OUT / f"test_images_{TAG}"), "zip", IMG_DIR)
lines = [f"# Notebook 03a: retry system on the low-memory model ({LOWMEM_SETUP})", "",
         f"Mode: {MODE_TAG}. {len(complete)} held-out test prompts from 02b, same seeds, same gate, same router, same judge.", "",
         "## Speed per level", "", speed.round(3).to_markdown(), "",
         f"Model size: {SIZE_MB:.0f} MB. " if SIZE_MB else "",
         f"App GPU memory (generator + gate): {LOADED_GB:.2f} GB loaded, {APP_PEAK_GB:.2f} GB peak." if APP_PEAK_GB else "", "",
         "## Strategies: float16 vs low-memory (judge: Mask2Former + CLIP ViT-L/14, GenEval-style)", "", SIDE.round(3).to_markdown(), "",
         f"**Verdict:** {VERDICT}", "",
         "## The app's modes (low-memory, checks after every try, time budgets applied)", "", MODES_TABLE.round(3).to_markdown(), "",
         "## Live requests (full pipeline incl. polish + WebP, no network)", "", LIVE.round(3).to_markdown(), ""]
(OUT / f"03a_summary_{TAG}.md").write_text("\n".join(lines), encoding="utf-8")
print("Saved: cascade_config_lowmem.json (for the app) to", SHARED, "| summary, CSVs, figures to", OUT)
if ON_KAGGLE:
    shutil.make_archive(str(KAGGLE / "03a_results"), "zip", OUT)
    print("Kaggle: everything is packed in 03a_results.zip (Output panel on the right). "
          "Unzip it into the repo as results/03a/ so notebook 03b can use it.")

## 12 · How to read this
- **The side-by-side table is the answer.** If "cascade from R0" (the app's Balanced mode) keeps its accuracy on the low-memory model (within ~3 points), the app's numbers are the low-memory ones, and the memory saving is real *and* free.
- **Small samples are noisy.** In QUICK mode each GenEval prompt is ~1% of the accuracy, so a 1–2 point change is noise.
- **Same seeds, not identical pictures.** Quantized weights change the pictures slightly (section 10 shows a few side by side).
- **The app's mode numbers include the check after every try**, the time budgets, and (live) the polish + WebP step. They don't include network time to the phone.
- **The router and the gate threshold are the ones trained on float16 pictures in 02b.** If the low-memory model's CLIP scores shift, the gate's pass rate shifts with it (the modes table shows the gate pass rate).